# Full Annotation Run: Agreement Ceiling + Task Pruning

1. **Human-human agreement** (ordinal Krippendorff's alpha + pairwise percent agreement, 50 items x 3 raters) — the ceiling any LLM-judge's human-AI agreement is measured against.
2. **Prune invalid tasks** where humans disagree widely (spread >= 2 on the 0-5 scale): if humans can't agree on a score, the item can't validate a judge.
3. Export the kept tasks with each rater's score and the panel median for the human-AI comparison.

In [11]:
import numpy as np
import pandas as pd

SCALE = list(range(0, 6))
K = len(SCALE)

df = pd.read_csv('full-annotator-scores.csv')
df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

wide = df.pivot(index='task_id', columns='tagger_name', values='score')
RATERS = list(wide.columns)
M = wide.to_numpy(dtype=float)
n_units, R = M.shape
print(f'{n_units} units x {R} raters, scale 0-{K-1}, missing cells: {int(np.isnan(M).sum())}')

50 units x 3 raters, scale 0-5, missing cells: 0


---
## 1. Human-human agreement (the ceiling)

Same estimator as the pilot: ordinal Krippendorff's alpha with a percentile bootstrap CI over units. At n=50 the point estimate is informative, unlike the pilot's n=10.

In [12]:
def krippendorff_alpha_ordinal(matrix, scale=SCALE):
    """Krippendorff's alpha with the ordinal difference function. matrix = units x raters."""
    pos = {c: i for i, c in enumerate(scale)}
    k = len(scale)

    o = np.zeros((k, k))
    for row in matrix:
        vals = [v for v in row if not np.isnan(v)]
        m = len(vals)
        if m < 2:
            continue
        for a in vals:
            for b in vals:
                o[pos[a], pos[b]] += 1.0 / (m - 1)
        for a in vals:
            o[pos[a], pos[a]] -= 1.0 / (m - 1)

    n_c = o.sum(axis=1)
    n = n_c.sum()
    if n < 2:
        return np.nan

    d2 = np.zeros((k, k))
    for c in range(k):
        for kk in range(k):
            lo, hi = min(c, kk), max(c, kk)
            d2[c, kk] = (n_c[lo:hi + 1].sum() - (n_c[c] + n_c[kk]) / 2) ** 2

    D_o = (o * d2).sum() / n
    D_e = (np.outer(n_c, n_c) * d2).sum() / (n * (n - 1))
    return 1 - D_o / D_e


def alpha_with_ci(matrix, n_boot=5000, seed=0):
    a = krippendorff_alpha_ordinal(matrix)
    rng = np.random.default_rng(seed)
    n = matrix.shape[0]
    boot = np.array([krippendorff_alpha_ordinal(matrix[rng.integers(0, n, n)])
                     for _ in range(n_boot)])
    lo, hi = np.nanpercentile(boot, [2.5, 97.5])
    return a, lo, hi, boot


alpha, lo, hi, boot = alpha_with_ci(M)
print(f'ordinal Krippendorff alpha = {alpha:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   (n={n_units}, R={R})')
print(f'P(alpha >= 0.80) = {np.mean(boot >= 0.80):.2f}   P(alpha >= 0.67) = {np.mean(boot >= 0.67):.2f}')

ordinal Krippendorff alpha = 0.775   95% CI [0.625, 0.874]   (n=50, R=3)
P(alpha >= 0.80) = 0.32   P(alpha >= 0.67) = 0.92


---
## 1b. Pairwise percent agreement

Raw (not chance-corrected) agreement, as an interpretable companion to alpha and a direct
ceiling for the same metrics computed human-vs-judge:
- `exact` — proportion of tasks where the pair gave identical scores
- `within_1` — proportion where the pair differed by at most 1 point

Mean over the 3 rater pairs is the headline number.

In [13]:
from itertools import combinations

def pairwise_agreement(scores):
    """Per-pair exact and +-1 agreement over a units x raters DataFrame, with a mean row."""
    p = pd.DataFrame([
        {'pair': f'{a} vs {b}',
         'exact': np.nanmean(scores[a] == scores[b]),
         'within_1': np.nanmean((scores[a] - scores[b]).abs() <= 1)}
        for a, b in combinations(scores.columns, 2)
    ]).set_index('pair')
    p.loc['— mean —'] = p.mean()
    return p


pairs = pairwise_agreement(wide)
print(f"mean pairwise exact agreement = {pairs.loc['— mean —', 'exact']:.3f}")
print(f"mean pairwise +-1 agreement   = {pairs.loc['— mean —', 'within_1']:.3f}")
pairs.round(3)

mean pairwise exact agreement = 0.580
mean pairwise +-1 agreement   = 0.933


,exact,within_1
pair,,
Jesi vs Rubayet Bushra,0.52,0.880
Jesi vs Yousuf,0.62,0.920
Rubayet Bushra vs Yousuf,0.60,1.000
— mean —,0.58,0.933


---
## 2. Prune invalid tasks (wide human disagreement)

A task is invalid when max - min score across the 3 raters >= 2: the raters do not share a reading of the item, so it cannot serve as a reference for judging an AI grader. Spread of 1 is ordinary adjacent-category noise on an ordinal scale and is kept.

In [14]:
SPREAD_CUT = 2

items = wide.copy()
items['median'] = np.nanmedian(M, axis=1)
items['spread'] = np.nanmax(M, axis=1) - np.nanmin(M, axis=1)
items['invalid'] = items['spread'] >= SPREAD_CUT

pruned = items[items['invalid']].sort_values('spread', ascending=False)
kept = items[~items['invalid']]
print(f'pruned {len(pruned)} / {n_units} tasks (spread >= {SPREAD_CUT}), {len(kept)} kept')
pruned

pruned 6 / 50 tasks (spread >= 2), 44 kept


tagger_name,Jesi,Rubayet Bushra,Yousuf,median,spread,invalid
task_id,,,,,,
T5-H3,1,5,4,4.0,4.0,True
T10-H1,4,2,3,3.0,2.0,True
T3-H3,3,5,5,5.0,2.0,True
T5-H1,2,0,0,0.0,2.0,True
T8-E2,3,5,5,5.0,2.0,True
T9-H3,0,2,1,1.0,2.0,True


In [15]:
# rater comments on the pruned tasks, for the write-up
notes = df.set_index(['task_id', 'tagger_name'])
for tid, row in pruned.iterrows():
    print('=' * 100)
    print(f'{tid}   median={row["median"]:.1f}  spread={int(row["spread"])}')
    for r in RATERS:
        print(f'  [{int(row[r])}] {r}: {notes.loc[(tid, r), "comments"]}')
    print()

T5-H3   median=4.0  spread=4
  [1] Jesi: Answers the question but the reasoning is fabricated
  [5] Rubayet Bushra: Relevant, correct verdict (No answer) and conclusion (test not reliable due to insufficient data).  All requested parts included. The agent's extra detail doesn't contradict the gold answer.
  [4] Yousuf: Correct direction and relevant but agent adding more information

T10-H1   median=3.0  spread=2
  [4] Jesi: Some of the numerical figures in the comparison point is inaccurate.
For the roadmap, there is no inclusion of numerical facts. It mentions suggestions not given in Gold answer
  [2] Rubayet Bushra: Relevant and accurate on overall sentiment and top 3 issues but the industry comparison as verdict is completely wrong. The agent states the difference is statistically significant when the gold states it is not (p=0.290), it also gives the wrong industry average. Counts for mentions also different from gold.
  [3] Yousuf: Right direction and relevant. But numbers are w

---
## 3. Ceiling on the kept set + export

Alpha and pairwise agreement recomputed on kept tasks only — these are the human-human ceilings the LLM-judge is compared against, since the judge will only be evaluated on these tasks. Kept tasks are written to `valid_tasks.csv` with each rater's individual score and the panel median.

In [16]:
M_kept = kept[RATERS].to_numpy(dtype=float)
alpha_k, lo_k, hi_k, boot_k = alpha_with_ci(M_kept)
pairs_k = pairwise_agreement(kept[RATERS])

print(f'all {n_units} tasks   : alpha = {alpha:.3f}   95% CI [{lo:.3f}, {hi:.3f}]')
print(f'kept {len(kept)} tasks  : alpha = {alpha_k:.3f}   95% CI [{lo_k:.3f}, {hi_k:.3f}]   <- human-human ceiling')
print(f"\nkept-set mean pairwise exact agreement = {pairs_k.loc['— mean —', 'exact']:.3f}")
print(f"kept-set mean pairwise +-1 agreement   = {pairs_k.loc['— mean —', 'within_1']:.3f}")

out = kept[RATERS + ['median']]
out.to_csv('valid_tasks.csv')
print(f"\nwrote {len(out)} tasks to valid_tasks.csv")
out.head()

all 50 tasks   : alpha = 0.775   95% CI [0.625, 0.874]
kept 44 tasks  : alpha = 0.830   95% CI [0.688, 0.907]   <- human-human ceiling

kept-set mean pairwise exact agreement = 0.636
kept-set mean pairwise +-1 agreement   = 1.000

wrote 44 tasks to valid_tasks.csv


tagger_name,Jesi,Rubayet Bushra,Yousuf,median
task_id,,,,
T1-E1,5,5,5,5.0
T1-E2,0,0,1,0.0
T1-H1,5,5,5,5.0
T1-H2,0,0,0,0.0
T1-H3,5,5,5,5.0
